# 第11回 機械学習による将棋AIの特徴分析

## 教師なし学習とクラスタリング

---

## 本講義の目標

本講義では，以下の4点を到達目標とする．

1. **教師なし学習の理解:** 正解データを持たないデータから構造を発見する「クラスタリング」の概念を学ぶ．
2. **K-Means法の習得:** 代表的なクラスタリング手法であるK-Means法のアルゴリズムを直感的に理解する．
3. **特徴量エンジニアリングの実践:** Pandasを用いた集計を通じ，機械学習に与える「意味のある指標」を構成する．
4. **データの解釈:** 分類した結果を可視化し，人間の視点で意味付けを行う．

---

## 1. 導入：機械学習の2つの顔

機械学習は，大きく分けて以下の2つに分類される．

* **教師あり学習（Supervised Learning）**
  * 「正解」データを与え，未知のデータに対する予測を行う．
  * 例：画像に写っている動物が犬か猫かを判定する．
* **教師なし学習（Unsupervised Learning）**
  * 「正解」を与えず，データそのものに潜む「構造」や「関係性」を発見する．
  * 例：似ている顧客をグループ分けする．

今回は，教師なし学習の代表的手法である**クラスタリング**を扱う．

---

## クラスタリングの目的と応用例

**クラスタリング**とは，似たもの同士を集めてグループ（クラスタ）を作ることである．

**【実社会での応用例】**
* **顧客セグメンテーション：** 購買履歴から，似たような傾向を持つ顧客層を分類し，ターゲットを絞った広告を打つ．
* **ニュースの自動分類：** 記事に含まれる単語の傾向から，政治・経済・スポーツなどのカテゴリに自動で分ける．
* **異常検知：** どのグループにも属さない「外れ値」を見つけることで，クレジットカードの不正利用などを検知する．

---

## 2. 直感的な理解：K-Means法の仕組み

クラスタリングの代表的なアルゴリズムが **K-Means法（K平均法）** である．
指定した「K個」のグループにデータを自動分類する．

**【陣取りゲームによる直感的な例え】**
1. 空間上に適当にK人の「仮のリーダー（重心）」を配置する．
2. すべてのデータは，自分から一番近いリーダーのチームに入る．
3. チームができた後，チーム全員の「ちょうど真ん中（平均）」の位置に，リーダーが移動する．
4. 新しいリーダーの位置で，再び2〜3を繰り返す．
5. リーダーが動かなくなったら分類完了である．

---

## シミュレータによるK-Meansの確認

実際のアルゴリズムがどのように収束していくか，シミュレータを用いて確認する．

In [ ]:
from IPython.display import IFrame

# 「青の統計学」のクラスタリング可視化ツール
url = "https://app.statisticsschool.com/tool/clustering-visualizer/"
display(IFrame(url, width=1000, height=800))

※Colab上で表示されない場合は，以下のリンクから確認すること．
[クラスタリング可視化ツール（青の統計学）](https://app.statisticsschool.com/tool/clustering-visualizer/)

---

## 3. 分析の準備：特徴量エンジニアリング

今回は，将棋AIたちをその「特徴」でクラスタリングする．
機械学習に与える「判断材料」を人間が設計することを**特徴量エンジニアリング**と呼ぶ．

今回は以下の3つの特徴量を用いる．
1. **先手勝率:** AIの純粋な「強さ」
2. **振り飛車率:** AIの「戦法の好み」
3. **手番ギャップ:** 「先手勝率 - 後手勝率」．後手番での頑健性（後手でも弱くならないか）を表す指標．

---

## データの計算と結合

前回学んだPandasの集計を活用し，各AIの特徴量を1つの表にまとめる．

In [ ]:
import pandas as pd
# データ読み込みと勝率・振り飛車率の計算（前回と同様の手順）
df = pd.read_csv('https://raw.githubusercontent.com/k0-1/data_science_2026/main/sessions/07_pandas/floodgate_data2026.csv')

次に対局数トップ20のAIについて先手勝率と後手勝率を計算する

In [ ]:
top20_ai = pd.concat([df['先手'], df['後手']]).value_counts().head(20).index # 対局数トップ20の将棋AIの集まり(Index object)
df_sente = df[df['先手'].isin(top20_ai)]
df_gote = df[df['後手'].isin(top20_ai)]

sente_win = pd.crosstab(df_sente['先手'], df_sente['勝者'], normalize='index')['先手'] # 先手勝率
gote_win = pd.crosstab(df_gote['後手'], df_gote['勝者'], normalize='index')['後手'] # 後手勝率

振り飛車率を計算する．
以下ではTrueを1, Falseを0として平均を計算している．

In [ ]:
furi_rate = df_sente.groupby('先手')['先手振り飛車'].mean()

データフレームの合体と手番ギャップの計算を行う．

In [ ]:
df_features = pd.DataFrame({'先手勝率': sente_win, '後手勝率': gote_win, '振り飛車率': furi_rate})
df_features['手番ギャップ'] = df_features['先手勝率'] - df_features['後手勝率']
display(df_features)

---

## 4. 実装：Pythonによるクラスタリング

機械学習ライブラリ`scikit-learn` を使い，K-Means法を実行する．今回は4つのグループに分類するよう（`n_clusters=4`）指示を出す．

In [ ]:
from sklearn.cluster import KMeans

# 1. 機械学習に渡す3つの特徴量（列）だけを抽出する
X = df_features[['先手勝率', '振り飛車率', '手番ギャップ']]

# 2. 4つのグループに分けるモデルを準備する
kmeans_model = KMeans(n_clusters=4, random_state=42)
# random_state=42は，実行のたびに分類結果（グループの番号や色）が変わるのを防ぐための「乱数固定」のおまじない（数字はなんでもよい）


# 3. 分類を実行し，結果（0〜3の番号）を元の表に追加する
df_features['グループ'] = kmeans_model.fit_predict(X)
display(df_features.head())

---

## 直感的な可視化：3D散布図
インタラクティブ（操作可能）なグラフを描画するライブラリ`Plotly`を使って，
3つの特徴量（X，Y，Z軸）を用いた分類結果を3D空間に可視化する．
マウスで回しながら，グループごとの傾向を観察できる．

In [ ]:
import plotly.express as px

df_features['グループ'] = df_features['グループ'].astype(str) # 色分けのため文字列化

fig_3d = px.scatter_3d(
    df_features, 
    x='先手勝率', y='振り飛車率', z='手番ギャップ',
    color='グループ',
    hover_name=df_features.index,
    title='将棋AIの特徴分析'
)
fig_3d.show()

グラフをHTML形式で保存するには次の`write_html(ファイル名)`とすればよい．

In [ ]:
# 3DグラフをHTML形式でダウンロード可能なファイルとして保存
fig_3d.write_html("shogi_ai_3d_profile.html")

---

## 5. 考察と課題：データの意味付け

機械学習はデータを自動で分けてくれるが，**分かれたグループが「何を意味しているか」を解釈するのは(今回は)人間の仕事**である．

### 【提出課題】
ファイル提出：対局数トップ40の将棋AIについて上と同様に4つのグループにクラスタリングを行い，3Dグラフを作成してください．
そのグラフをHTML形式で保存し，提出してください．

テキスト提出：

0. 得られた40個のAIの4つのグループ分けについて，各グループに「キャッチコピー」をつけ，そう名付けた理由を説明してください（堅苦しく考える必要はありません）．
例: グループ１：「強い振り飛車党」，理由： 先手勝率の高い振り飛車党が集まっているので．

1. 今回の講義の感想・コメント・疑問点を書いてください．








